Step 0 — Pick a domain with two related resources. CRUD on a single table teaches half the lesson. Something like notes + tags, courses + enrollments, or (feeding forward nicely) ML experiments + runs gives you a one-to-many relationship, which is where real API design questions show up. Keep it to two tables — three is scope creep on a one-day build.


Step 1 — Skeleton first, database later. Get a walking skeleton running in the first half hour with an in-memory dict as your "database":


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class NoteIn(BaseModel):
    title: str
    body: str = ""

@app.post("/notes")
def create_note(note: NoteIn):
    ...

Run it with uvicorn main:app --reload and open /docs — FastAPI generates an interactive Swagger UI automatically, and clicking around it beats curl for fast iteration. The key habit to establish immediately: separate Pydantic models for input (NoteIn) and output (NoteOut with id and created_at). Letting clients set their own IDs is the classic beginner hole.Run it with uvicorn main:app --reload and open /docs — FastAPI generates an interactive Swagger UI automatically, and clicking around it beats curl for fast iteration. The key habit to establish immediately: separate Pydantic models for input (NoteIn) and output (NoteOut with id and created_at). Letting clients set their own IDs is the classic beginner hole.


Step 2 — Swap in Postgres. Start Postgres the easy way (docker run -e POSTGRES_PASSWORD=dev -p 5432:5432 postgres — a preview of the next challenge), then wire it up with SQLAlchemy 2.0 style: one models.py for tables, one database.py for the engine and session factory, and a get_db dependency that yields a session per request. That dependency-injection pattern (db: Session = Depends(get_db)) is the FastAPI idea — the same mechanism will handle auth in the next step. Read the connection string from an environment variable, never hardcode it; Tier 2's Docker challenge depends on this. Beginners can use SQLite here instead and still complete the challenge — the SQLAlchemy code is nearly identical, which is the point of an ORM.


Step 3 — CRUD endpoints with correct semantics. The five endpoints are mechanical; the craft is in the details that distinguish a real API: return 201 on create and 404 (via HTTPException) when an ID doesn't exist — never let it become a 500; decide PUT (full replace) vs PATCH (partial update) and be consistent; return 204 with no body on delete. Test the unhappy paths in /docs as you go: fetching a deleted note, sending a string where a number belongs, missing required fields.


Step 4 — Auth, the honest version. Full OAuth2 with JWTs is a day by itself, so tier it. Base version: static API keys — a verify_key dependency that reads an X-API-Key header and compares against a table (or even an env var), applied to all write endpoints via Depends. That's real auth, used by real services. Ambitious folks do the JWT flow instead: a /token endpoint checking a hashed password (passlib with bcrypt) and issuing a signed JWT (python-jose), plus a get_current_user dependency that decodes it. The non-negotiables either way: passwords stored only as hashes, and a 401 with a clear message on bad credentials. A nice touch for the one-to-many design: notes belong to the authenticated user, and users can only see their own — that turns auth from a gate into actual data modeling.


Stretch part 1 — Pagination. Add limit and offset query parameters with sane bounds (limit: int = Query(20, le=100)), and return a wrapper object — {"items": [...], "total": 123, "limit": 20, "offset": 40} — rather than a bare list, so clients can render page controls. The deeper cut, if someone wants it: cursor-based pagination (?after_id=57), and being able to explain why offset pagination breaks when rows are inserted mid-scroll.


Stretch part 2 — Rate limiting. The slowapi library gets you decorator-based limits (@limiter.limit("10/minute")) in twenty minutes. The interesting engineering question to discuss even if you don't build it: the in-process counter dies when you run two replicas — which is why production rate limiting lives in Redis, and which is exactly the "Queue it up" challenge's infrastructure making a second appearance.


Stretch part 3 — OpenAPI polish. This one's underrated for job interviews: add summary, description, and response_model to every route, Field(description=..., examples=[...]) to Pydantic models, and tags to group endpoints. Then open /docs and judge it by one standard — could a stranger use this API without asking you anything? That page is effectively your API's README.
